In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pathlib import Path
# read in all the data for yellow taxis
spark = SparkSession.builder.appName("Read Parquet") \
    .config("spark.driver.memory", "12g").getOrCreate()

YELLOW_DIR = Path('data/year-2026-yellow/')
# read in all the data across the times
from pathlib import Path

data_root = Path("data")
pattern = "year-*-yellow"  # matches year-2026-yellow, year-2025-yellow, etc.
dirs = sorted([d for d in data_root.glob(pattern) if d.is_dir()])

parquet_files = []
for d in dirs:
    parquet_files.extend(
        sorted([p for p in d.glob("*.parquet") if p.is_file()])
    )

yellow_data = None
for p in parquet_files:
    df = spark.read.parquet(str(p))
    if yellow_data is None:
        yellow_data = df
    else:
        # use unionByName to be robust to column ordering / missing columns
        yellow_data = yellow_data.unionByName(df, allowMissingColumns=True)

# yellow_data is the big row-wise union of all parquet files
yellow_data.printSchema()

# print dataset shape
print('Shape: ', yellow_data.count(), ',', len(yellow_data.columns))

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Shape:  147201830 , 20


# Cleaning the dataset
## Objective: Remove all irrelevant rows and columns
We remove select rows where certain column values are deemed infeasible, unrealistic or not representative of the typical circumstances
surrounding yellow taxi trips. We remove certain columns if they are not useful to our analysis. 
Some rows may have unusual values in certain columns due to reasons such as recording mistakes (driver forgetting to reset system), corrupt data or system glitches. 

<b>Rows are to be removed for the following reasons</b>:
- Passenger count is 0 or null
- Trip distance is 0, null or greater than a reasonable threshold like 25 miles. This filters out trips that are unusual in their distance, likely due to errors in the recording system or highly exceptional circumstances.
- Trip durations are unrealistic (e.g a few seconds or too many hours)
- Drop off or pickup area is unknown, outside NYC or an area that motor vehicles cannot travel to

<b> The following columns will be retained with all others removed</b>:
- `PULocationID`: TLC ID for zone where pickup occurred. 
- `total_amount`: Total fare charged to the passengers
- `tpep_pickup_datetime`: Time that taxi picked up passengers

In [4]:
# get rid of rows where passenger count is 0 or null
yellow_data = yellow_data.filter((F.col("passenger_count") > 0) & 
                                 (F.col("passenger_count").isNotNull()))
# get rid of rows where trip distance is 0, null or greater than 25
yellow_data = yellow_data.filter((F.col("trip_distance") > 0) & 
                                 (F.col("trip_distance").isNotNull()) & 
                                 (F.col("trip_distance") <= 25))
# get rid of rows with undesirable trip durations
yellow_data = yellow_data.filter((F.col("tpep_dropoff_datetime").isNotNull()) 
                                 & (F.col("tpep_pickup_datetime").isNotNull()))
yellow_data = yellow_data.withColumn("trip_duration", 
    F.unix_timestamp("tpep_dropoff_datetime") 
    - F.unix_timestamp("tpep_pickup_datetime")
)
yellow_data = yellow_data.filter((F.col("trip_duration") > 60) 
                                 & (F.col("trip_duration").isNotNull()) 
                                 & (F.col("trip_duration") < 10000))
junk_locations = [264, 265, 103, 104, 110]
yellow_data = yellow_data.filter(~F.col("PULocationID").isin(junk_locations))
# get rid of rows where payment_type is null or in the array [5, 6]
yellow_data = yellow_data.filter((F.col("payment_type").isNotNull()) 
                                 & (~F.col("payment_type").isin([5, 6])))
# remove rows where the year in tpep_pickup_datetime is not in range 2023-2026
yellow_data = yellow_data.withColumn(
    "pickup_year", F.year("tpep_pickup_datetime")
)
yellow_data = yellow_data.filter(
    (F.col("pickup_year") >= 2023) & (F.col("pickup_year") <= 2026)
)

# drop unnecessary columns
columns_to_retain_indices = [1, 7, 16]
yellow_data = yellow_data.select([yellow_data.columns[i] for i in columns_to_retain_indices])

# print shape of the cleaned data
print('Shape: ', yellow_data.count(), ",", len(yellow_data.columns))

Shape:  120193181 , 3


In [5]:
# save the cleaned data to a new parquet file
yellow_data.write.parquet("cleaned_data/yellow_data.parquet", mode="overwrite")

# Cleaning the FHVHV dataset
## Objective: Remove rows and columns for the same reasons as with the yellow taxi dataset

<b>The columns we will retain are expected to tell the equivalent information (or identical) as the yellow taxi trip data. These are</b>:
- `pickup_datetime`: Time that vehicle picked up passengers.
- `PULocationID`: TLC ID for taxi zone where pickup occurred.
- `driver_pay`: Total money paid to drivers not including tolls, tips, surcharges or commission.

In [ ]:
pattern = "year-*-fhvhv"  # matches year-2026-yellow, year-2025-yellow, etc.
dirs = sorted([d for d in data_root.glob(pattern) if d.is_dir()])

parquet_files = []
for d in dirs:
    parquet_files.extend(
        sorted([p for p in d.glob("*.parquet") if p.is_file()])
    )

fhvhv_data = None
for p in parquet_files:
    df = spark.read.parquet(str(p))
    if fhvhv_data is None:
        fhvhv_data = df
    else:
        # use unionByName to be robust to column ordering / missing columns
        fhvhv_data = fhvhv_data.unionByName(df, allowMissingColumns=True)

# fhvhv_data is the big row-wise union of all parquet files
print(f"Combined {len(parquet_files)} files. {fhvhv_data.count()} rows")
fhvhv_data.printSchema()

# print the dataset shape
print('Shape: ', fhvhv_data.count(), ',', len(fhvhv_data.columns))

Combined 41 parquet files into dataframe with 821546266 rows
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- sha

In [ ]:
# data cleaning
# remove rows where driver_pay is null, less than or equal to 0
fhvhv_data = fhvhv_data.filter(fhvhv_data.driver_pay.isNotNull() 
                               & (fhvhv_data.driver_pay > 0))
# remove rows where trip time is null or less than 10 seconds
fhvhv_data = fhvhv_data.filter(fhvhv_data.trip_time.isNotNull() 
                               & (fhvhv_data.trip_time >= 10))
# remove rows where trip miles is null or 0
fhvhv_data = fhvhv_data.filter(fhvhv_data.trip_miles.isNotNull() 
                               & (fhvhv_data.trip_miles > 0))
# remove rows where PULocationID is in the list of invalid IDs
fhvhv_data = fhvhv_data.filter(~F.col("PULocationID").isin(junk_locations))

columns_to_retain_indices = [5, 7, 18]

fhvhv_data = fhvhv_data.select([fhvhv_data.columns[i] for i in columns_to_retain_indices])

print('Shape: ', fhvhv_data.count(), ',', len(fhvhv_data.columns))

In [7]:
# save the cleaned data to a new parquet file
fhvhv_data.write.mode("overwrite").parquet("cleaned_data/fhvhv_data.parquet")

26/08/26 23:22:00 WARN DAGScheduler: Broadcasting large task binary with size 1192.4 KiB
